## Time Series & Panel Dataset Processing
This module provides a functional model to create a simple times series dataset measuring affordability over time. This measure of affordability uses the average price per square meter dataset and the median gross hourly wage dataset. Both datasets observe these statistics for the 106 different provinces(NUTS3) in Italy.

The data sources are:
1) Idealista - Average Rental Price per Square Meter
2) ISTAT - Median gross hourly wage of full-time workers

Additional statistics include the following: 
* The tax standard income tax rate of 23%.
* The monthly number of hours of a full-time worker.
* The average or median size of an apartment in square meters.

The affordability dependent variable will be constructed as follows:

$$  Affordability = \frac{Rent_{i}*Apartment size}{Income_{i}*(Hours worked)}  $$

The Regression equation will be expressed as follows:

$$ Y_{i} = \alpha + \beta_{i}x_{i}$$

* $Y_{i}$ = Affordabilty % or how much of a median person's monthly income is spent on a standard apartment
* $\alpha$ = Intercept
* $\beta_{i}$ = Coefficient of affordability change
* $x_{i}$ = Year

## Step 1: Load libraries and datasets
Start by loading all of the libraries that may be used in the dataset.

In [95]:
import pandas as pd
import numpy as np

Load the rents and wages datasets first before merging them. Make sure they both have 106 observations and verify that there are no missing variables.

In [96]:
df_rent = pd.read_csv("Data/rent_prices.csv".lower())

# Strip whitespace from column names
df_rent.columns = df_rent.columns.str.strip()

# Strip whitespace from the Province column values
df_rent["Province"] = df_rent["Province"].str.strip()

df_rent.info() #Verify rent data is loaded correctly

<class 'pandas.DataFrame'>
RangeIndex: 107 entries, 0 to 106
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Province  107 non-null    str    
 1   2014      107 non-null    float64
 2   2015      107 non-null    float64
 3   2016      107 non-null    float64
 4   2017      107 non-null    float64
 5   2018      107 non-null    float64
 6   2019      107 non-null    float64
 7   2020      107 non-null    float64
 8   2021      107 non-null    float64
 9   2022      107 non-null    float64
 10  2023      107 non-null    float64
dtypes: float64(10), str(1)
memory usage: 9.3 KB


In [97]:
df_wages = pd.read_csv("Data/median_gross_wages.csv".lower(),delimiter=";", encoding="latin-1")

# Strip whitespace from column names
df_wages.columns = df_wages.columns.str.strip()

# Strip whitespace from the Time column values
df_wages["Time"] = df_wages["Time"].str.strip()

df_wages.info() #Verify wages data is loaded correctly

<class 'pandas.DataFrame'>
RangeIndex: 107 entries, 0 to 106
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Time    107 non-null    str    
 1   2014    107 non-null    float64
 2   2015    107 non-null    float64
 3   2016    107 non-null    float64
 4   2017    107 non-null    float64
 5   2018    107 non-null    float64
 6   2019    107 non-null    float64
 7   2020    107 non-null    float64
 8   2021    107 non-null    float64
 9   2022    107 non-null    float64
 10  2023    107 non-null    float64
dtypes: float64(10), str(1)
memory usage: 9.3 KB


In [98]:
# Sort Provines alphabetically and change Time to Province for easier merging later on
df_rent = df_rent.rename(columns={"Time":"Province"})  
df_rent = df_rent.sort_values(by="Province")

df_rent.head(len(df_rent)-10)

,Province,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,Agrigento,5.1,5.0,4.9,5.3,4.8,5.2,5.1,6.1,5.7,6.3
1,Alessandria,5.0,5.2,5.3,5.3,5.3,5.1,5.3,5.4,5.4,5.6
2,Ancona,7.1,6.9,7.1,7.1,6.8,7.2,7.3,7.4,7.9,8.4
3,Aosta,10.5,10.2,8.8,9.8,11.8,10.6,10.7,13.2,15.5,19.1
4,Arezzo,7.0,7.1,6.8,7.1,6.7,7.1,7.2,7.1,7.7,8.5
...,...,...,...,...,...,...,...,...,...,...,...
92,Terni,5.7,5.7,5.6,5.3,5.2,5.3,5.7,5.8,5.9,6.5
93,Torino,7.8,7.5,7.2,7.6,7.8,7.4,7.9,8.0,9.1,10.4
94,Trapani,6.3,5.8,5.4,5.8,5.5,5.3,5.4,5.7,5.9,6.9
95,Trento,8.7,9.0,8.9,9.4,9.3,10.3,10.4,11.1,12.7,12.7


## Step 2: Defining values and functions

In [99]:
#Function to adjust wages for tax based on 23% for the first 15,000 euros, 27% up to 28,000 euros (or 25% in 2022 and 2023), and 38% for anything above 28,000 euros (or 35% in 2022 and 2023)

def Tax_Adjusted_Wage(wage, year):  
    # Convert year to string if it's an integer
    year = str(year)
    wage = wage*170*12
    wage = float(wage) #Convert wage to integer for easier calculations
    if year == "2022" or year == "2023": #For years 2022 and 2023
        if wage <= 15000:
            wage = wage - (wage*0.23) 
        elif wage <= 28000 and wage > 15000:
            wage = wage - (15000*0.23 + (wage-15000)*0.25)
        else: 
            wage = wage - (15000*0.23 + 13000*0.25 + (wage-28000)*0.35)
    else:
        if wage <= 15000:
            wage = wage - (wage*0.23)
        elif wage <= 28000 and wage > 15000:
            wage = wage - (15000*0.23 + (wage-15000)*0.27)
        else:
            wage = wage - (15000*0.23 + 13000*0.27 + (wage-28000)*0.38)

    wage = wage/12 

    return round(wage, 2)

example_wage = 14.67 #Test wage to demonstrate function.

example_wage_2022 = Tax_Adjusted_Wage(example_wage, 2022)
example_wage_2018 = Tax_Adjusted_Wage(example_wage, 2018)

print(f"For a wage of €{example_wage} per hour, the tax-adjusted annual wage in 2022 would be €{example_wage_2022}, while in 2018 it would be €{example_wage_2018}.")

For a wage of €14.67 per hour, the tax-adjusted annual wage in 2022 would be €1879.37, while in 2018 it would be €1852.88.


In [100]:
for col in df_wages.columns[1:]: #Apply tax adjustment to all wage columns, skipping the first column which contains province names
    df_wages[col] = df_wages[col].apply(lambda x: Tax_Adjusted_Wage(x, col))

In [101]:
df_wages.head()

,Time,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,Torino,1657.10,1663.30,1662.06,1660.82,1652.13,1684.40,1724.11,1683.16,1709.28,1759.00
1,Vercelli,1529.27,1549.13,1562.78,1565.26,1560.30,1577.67,1643.44,1600.01,1646.80,1674.85
2,Novara,1557.81,1576.43,1581.39,1581.39,1583.88,1613.66,1633.52,1613.66,1648.07,1687.60
3,Cuneo,1523.07,1537.96,1555.33,1528.03,1536.72,1561.54,1573.95,1582.63,1612.38,1657.00
4,Asti,1494.52,1525.55,1539.20,1528.03,1535.48,1560.30,1576.43,1576.43,1608.55,1640.43


## Step 3: Create the Time Series dataset
This step will define the function and then apply it to the datasets to create the dependent variable. This will then be saved as a new dataset.

In [102]:
# Create a mapping to standardize province names between df_wages and df_rent
province_mapping = {
    "Bolzano / Bozen": "Bolzano-Bozen",
        "Bolzano": "Bolzano-Bozen",
        "Forlï¿½-Cesena": "Forlì-Cesena",
        "Monza e della Brianza": "Monza-Brianza",
        "Pesaro e Urbino": "Pesaro Urbino",
        "Reggio di Calabria": "Reggio Calabria",
        "Reggio nell'Emilia": "Reggio Emilia",
        "Valle d'Aosta": "Aosta",
        "Valle d'Aosta / Vallï¿½e d'Aoste": "Aosta",
        "Valle d'Aosta / Vall�e d'Aoste": "Aosta",
        "Valle d'Aosta / Vallée d'Aoste": "Aosta",
        "Valle d'Aosta / Vallée d'Aoste"  : "Aosta"
}

# Apply the mapping to df_wages
df_wages["Time"] = df_wages["Time"].replace(province_mapping)

In [103]:
def createDV(wage, rent, size): #Create dependent variable by dividing the cost of monthly rent (rent*size) by the net monthly wage
    dv = (rent*size)/wage
    return round(dv, 2)

In [104]:
# Calculate affordability index
df_affordability = df_wages.copy()
rent_size = 80 # Average size of rental apartments

for col in df_affordability.columns[1:]:
    df_affordability[col] = df_affordability.apply(lambda x: createDV(x[col], df_rent[df_rent["Province"] == x["Time"]][col].values[0], size=rent_size), axis=1)

df_affordability = df_affordability.sort_values(by="Time")
df_affordability.head()

,Time,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
96,Agrigento,0.30,0.29,0.28,0.30,0.27,0.29,0.28,0.34,0.31,0.34
5,Alessandria,0.26,0.26,0.27,0.27,0.27,0.26,0.26,0.27,0.26,0.26
60,Ancona,0.38,0.36,0.37,0.38,0.37,0.38,0.38,0.39,0.41,0.42
8,Aosta,0.57,0.55,0.48,0.53,0.63,0.57,0.56,0.70,0.81,0.99
53,Arezzo,0.40,0.39,0.38,0.39,0.37,0.39,0.39,0.38,0.41,0.44


In [105]:
df_affordability.info()

<class 'pandas.DataFrame'>
Index: 107 entries, 96 to 64
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Time    107 non-null    str    
 1   2014    107 non-null    float64
 2   2015    107 non-null    float64
 3   2016    107 non-null    float64
 4   2017    107 non-null    float64
 5   2018    107 non-null    float64
 6   2019    107 non-null    float64
 7   2020    107 non-null    float64
 8   2021    107 non-null    float64
 9   2022    107 non-null    float64
 10  2023    107 non-null    float64
dtypes: float64(10), str(1)
memory usage: 10.0 KB


In [106]:
output_file = "Data/affordability_index.csv"

df_affordability.to_csv(output_file, index=False, encoding="utf-8")
#df_wages.to_csv(output_file, path_or_buf="../Data/".lower(), index=False, encoding="utf-8")

## Step 4: Transform into Panel Data

In [107]:
df_afford = pd.read_csv("Data/affordability_index.csv")

In [108]:
# Transform from wide format to long format (panel data)
panel_data = df_afford.melt(
    id_vars=['Time'],
    var_name='Year',
    value_name='Affordability Index'
)

panel_data = panel_data.rename(columns={'Time': 'Province'}) # Rename columns for clarity
panel_data['Year'] = panel_data['Year'].astype(int)

# Sort by Province and Year for better organization
panel_data = panel_data.sort_values(['Province', 'Year']).reset_index(drop=True)


In [109]:
panel_data["Ln affordability Index"] = np.log(panel_data["Affordability Index"])

In [110]:
panel_data.head(10)

,Province,Year,Affordability Index,Ln affordability Index
0,Agrigento,2014,0.30,-1.203973
1,Agrigento,2015,0.29,-1.237874
2,Agrigento,2016,0.28,-1.272966
3,Agrigento,2017,0.30,-1.203973
4,Agrigento,2018,0.27,-1.309333
5,Agrigento,2019,0.29,-1.237874
6,Agrigento,2020,0.28,-1.272966
7,Agrigento,2021,0.34,-1.078810
8,Agrigento,2022,0.31,-1.171183
9,Agrigento,2023,0.34,-1.078810


In [111]:
panel_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1070 entries, 0 to 1069
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Province                1070 non-null   str    
 1   Year                    1070 non-null   int64  
 2   Affordability Index     1070 non-null   float64
 3   Ln affordability Index  1070 non-null   float64
dtypes: float64(2), int64(1), str(1)
memory usage: 33.6 KB


## Step 5: Geographical Features

This part is dedicated to adding a column for the regions as a categorical variable. This will not only make it easier to identify the location of provinves, but will also be necessary for NUTS2 specific variables.

In [112]:
regions_CSV = pd.read_csv("Data/provinces-regions.csv", encoding = "utf-8")
# clean column names by stripping whitespace

regions_CSV.columns = regions_CSV.columns.str.strip()
regions_CSV.info()

<class 'pandas.DataFrame'>
RangeIndex: 107 entries, 0 to 106
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Province  107 non-null    str  
 1   Region    107 non-null    str  
dtypes: str(2)
memory usage: 1.8 KB


In [113]:
def merge_regions(panel_data, regions_CSV):
    # ensure region column name is normalized
    regions_CSV = regions_CSV.rename(columns=lambda x: x.strip())
    regions_CSV["Region"] = regions_CSV["Region"].str.strip() 

    panel_data = panel_data.merge(
        regions_CSV[['Province', 'Region']], 
        on='Province', 
        how='left'
    )
    
    return panel_data

# Apply the function

panel_data = merge_regions(panel_data, regions_CSV)

# inspect results
panel_data.head(10)


,Province,Year,Affordability Index,Ln affordability Index,Region
0,Agrigento,2014,0.30,-1.203973,Sicilia
1,Agrigento,2015,0.29,-1.237874,Sicilia
2,Agrigento,2016,0.28,-1.272966,Sicilia
3,Agrigento,2017,0.30,-1.203973,Sicilia
4,Agrigento,2018,0.27,-1.309333,Sicilia
5,Agrigento,2019,0.29,-1.237874,Sicilia
6,Agrigento,2020,0.28,-1.272966,Sicilia
7,Agrigento,2021,0.34,-1.078810,Sicilia
8,Agrigento,2022,0.31,-1.171183,Sicilia
9,Agrigento,2023,0.34,-1.078810,Sicilia


In [114]:
regions_missing = panel_data.isnull()
print(f"Number of missing regions: {regions_missing['Region'].sum()}")

Number of missing regions: 0


## Step 6: Adding Features

This section will write functions for applying features for a specific province/region in a given year. Then the datasets will be imported and applied as necessary. 

In [115]:
def add_VAR_province(panel_data, new_data, var_name): # Take variables from new data and match by province & year of panel data
    new_data = new_data.rename(columns=lambda x: x.strip())
    new_data = new_data.rename(columns={new_data.columns[0]: "Province"})
    new_data["Province"] = new_data["Province"].str.strip()
    new_data["Province"] = new_data["Province"].replace(province_mapping)
    
    new_data = new_data.melt(
        id_vars=['Province'],
        var_name='Year',
        value_name=str(var_name)
        ) # Transform to long format so values are in a single column
    
    # Convert Year to integer to match panel_data
    new_data['Year'] = new_data['Year'].astype(int)

    panel_data = panel_data.merge(
        new_data[['Province', 'Year', var_name]], 
        on=['Province', 'Year'], 
        how='left'
        )
    
    return panel_data


def add_VAR_region(panel_data, new_data, var_name): # Same as above but match by region
    new_data = new_data.rename(columns=lambda x: x.strip())
    new_data = new_data.rename(columns={new_data.columns[0]: "Region"})
    new_data["Region"] = new_data["Region"].str.strip()
    
    new_data = new_data.melt(
        id_vars=['Region'],
        var_name='Year',
        value_name=str(var_name)
        ) # Transform to long format so values are in a single column
    
    # Convert Year to integer to match panel_data
    new_data['Year'] = new_data['Year'].astype(int)

    panel_data = panel_data.merge(
        new_data[['Region', 'Year', var_name]], 
        on=['Region', 'Year'], 
        how='left'
        )
    
    return panel_data

In [116]:
panel_data = add_VAR_province(panel_data, df_wages, "Median net stipend")
#Multiply rents by apartment size
df_rent2 = df_rent.copy()
for col in df_rent2.columns[1:]:
    df_rent2[col] = df_rent2[col]*rent_size
    df_rent2[col] = df_rent2[col].astype(int)

panel_data = add_VAR_province(panel_data, df_rent2, "Average rent")

In [117]:
def add_year_dummy(panel_data, year=[], column_name=None, prefix="year"):
    """Add a dummy indicator column that is 1 when Year == year, otherwise 0."""
    col = column_name or f"{prefix}_{year}"
    panel_data[col] = (
        panel_data["Year"].astype(str)
        .isin([str(y) for y in year])
        ).astype(int)
    return panel_data


def add_province_dummy(panel_data, province=[], column_name=None, prefix="province"):
    """Add a dummy indicator column that is 1 when Province is in the given list, otherwise 0."""
    if isinstance(province, str):
        province = [province]  # Convert single string to list
    col = column_name or f"{prefix}_{'_'.join(str(p).replace(' ', '_') for p in province)}"
    panel_data[col] = (
        panel_data["Province"].astype(str)
        .str.strip()
        .str.lower()
        .isin([str(p).strip().lower() for p in province])
    ).astype(int)
    return panel_data


In [118]:
def transform_float(dataframe):
    for col in dataframe.columns[1:]:
            dataframe[col] = dataframe[col].astype(str).str.strip().str.replace(",", "", regex=False)
            dataframe[col] = dataframe[col].astype(str).str.strip().str.replace(".", "", regex=False)
            dataframe[col] = pd.to_numeric(dataframe[col], errors='coerce').fillna(0).astype(float)
    return dataframe

def transform_int(dataframe):
    for col in dataframe.columns[1:]:
            dataframe[col] = dataframe[col].astype(str).str.strip().str.replace(",", "", regex=False)
            dataframe[col] = dataframe[col].astype(str).str.strip().str.replace(".", "", regex=False)
            dataframe[col] = pd.to_numeric(dataframe[col], errors='coerce').fillna(0).astype(int)
    return dataframe

def transform_data(dataframe):
    dataframe.columns = dataframe.columns.str.strip()
    dataframe = dataframe.rename(columns=lambda x: x.strip())

    if dataframe.columns[0] in dataframe.columns:
        dataframe[dataframe.columns[0]] = dataframe[dataframe.columns[0]].replace(province_mapping)
    
    return dataframe

In [119]:
#Drop early years and some provinces in Sardegna with missing data for some variables

panel_data = panel_data[~panel_data["Year"].isin([2014, 2015, 2016, 2017])]
panel_data = panel_data[~panel_data["Province"].isin(["Olbia-Tempio", "Ogliastra", "Medio Campidano", "Carbonia-Iglesias"])]

In [120]:
# Add dummy variables for COVID and Post-COVID periods
# panel_data = add_year_dummy(panel_data, [2020], column_name="COVID")
# panel_data = add_year_dummy(panel_data, [2020, 2021], column_name="COVID Recovery")
# panel_data = add_year_dummy(panel_data, [2022,2023], column_name="Post-COVID")

# List of top 10 cities by population
#city_list = ["Roma", "Milano", "Torino", "Napoli", "Palermo", "Firenze", "Bologna", "Genova", "Bari", "Catania"]

#panel_data = add_province_dummy(panel_data, city_list, column_name="Urban")

In [121]:
inflation_dictionary = {
    "2014" : 0.0,
    "2015" : 0.1, 	
    "2016" : -0.1,
    "2017" : 1.3, 	
    "2018" : 1.2,
    "2019" : 0.6,
    "2020" : -0.1,
    "2021" : 1.9,
    "2022" : 8.7,
    "2023" : 5.9,
}# Add inflation rate to panel_data based on the year
#panel_data['Inflation Rate'] = panel_data['Year'].astype(str).map(inflation_dictionary)

In [122]:
df_mortgage_rates = pd.read_csv("Data/ECB Data Portal_20260403173908.csv".lower(), encoding='utf-8')

df_mortgage_rates = df_mortgage_rates.drop(columns = "DATE", axis = 0)

# Initialize dictionary to store annual averages
mortgage_rates_dict = {}
year_list = [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]

# Get the column names (should be "TIME PERIOD" and "Interest rates for Mortgages")
year_col = df_mortgage_rates.columns[0]  # "TIME PERIOD" column
rate_col = df_mortgage_rates.columns[1]  # "Interest rates for Mortgages" column

# Iterate through each year
for year in year_list:
    # Filter rows where TIME PERIOD equals the year
    year_data = df_mortgage_rates[df_mortgage_rates[year_col] == year]
    
    # Collect all rates for this year
    if len(year_data) > 0:
        mortgage_list = year_data[rate_col].tolist()
        mortgage_sums = sum(mortgage_list)
        
        # Calculate average rate for the year by dividing sum by count
        average_rate = mortgage_sums / len(mortgage_list)
        mortgage_rates_dict[year] = round(average_rate, 4)


panel_data['Mortgage rate'] = panel_data['Year'].astype(float).map(mortgage_rates_dict)
panel_data['Mortgage rate'] = (panel_data['Mortgage rate']/100).round(4)
panel_data.head()

,Province,Year,Affordability Index,Ln affordability Index,Region,Median net stipend,Average rent,Mortgage rate
4,Agrigento,2018,0.27,-1.309333,Sicilia,1410.14,384,0.0666
5,Agrigento,2019,0.29,-1.237874,Sicilia,1428.75,416,0.0659
6,Agrigento,2020,0.28,-1.272966,Sicilia,1446.12,408,0.0629
7,Agrigento,2021,0.34,-1.078810,Sicilia,1446.12,488,0.0636
8,Agrigento,2022,0.31,-1.171183,Sicilia,1482.32,456,0.0695


In [123]:
pop_density = pd.read_csv("Data/n3_population_density.csv".lower(), encoding='utf-8')
rmt_wrk = pd.read_csv("Data/n2_remote_work_panel.csv".lower(), encoding='utf-8')
tourism_rental = pd.read_csv("Data/n3_tourism_rental_per_capita.csv", encoding="utf-8")
unemp_rate = pd.read_csv("Data/n3_unemployment_rate.csv".lower(), delimiter= ";", encoding='utf-8')
tot_mig = pd.read_csv("Data/total_migration.csv".lower(), delimiter=';', encoding='utf-8')
pop = pd.read_csv("Data/population_merged.csv".lower(), encoding='utf-8')

var_list = [(pop_density, "Population density"),
            (tourism_rental, "Short-term rental per capita"),
            (tot_mig, "Total Migration per 1k"),
            (pop, "Population"),
            (unemp_rate, "Unemployment rate")
            ]

In [124]:
panel_data["Province"] = panel_data["Province"].astype("category")
panel_data["Region"] = panel_data["Region"].astype("category")

north = ["Alessandria", "Asti", "Biella", "Cuneo", "Novara", "Torino", "Verbano-Cusio-Ossola", "Vercelli", 
    "Aosta", "Genova", "Imperia", "La Spezia", "Savona", "Bergamo", "Brescia", 
    "Como", "Cremona", "Lecco", "Lodi", "Mantova", "Milano", "Monza-Brianza", "Pavia", "Sondrio", "Varese",
    "Bolzano-Bozen", "Trento", "Belluno", "Padova", "Rovigo", "Treviso", "Venezia", "Verona", "Vicenza", 
    "Gorizia", "Pordenone", "Trieste", "Udine", "Bologna", "Ferrara", "Forlì-Cesena", "Modena", "Parma", 
    "Piacenza", "Ravenna", "Reggio Emilia", "Rimini"
    ]

centre = ["Arezzo", "Firenze", "Grosseto", "Livorno", "Lucca", "Massa-Carrara", "Pisa", "Pistoia", "Prato", "Siena", 
    "Perugia", "Terni", "Ancona", "Ascoli Piceno", "Fermo", "Macerata", "Pesaro Urbino", "Frosinone", 
    "Latina", "Rieti", "Roma", "Viterbo"]

island = ["Agrigento", "Caltanissetta", "Catania", "Enna", "Messina", "Palermo", "Ragusa", "Siracusa", "Trapani", 
    "Cagliari", "Nuoro", "Oristano", "Sassari", "Sud Sardegna"]

south = ["Chieti", "L'Aquila", "Pescara", "Teramo", "Campobasso", "Isernia", "Avellino", "Benevento", "Caserta", 
    "Napoli", "Salerno", "Bari", "Barletta-Andria-Trani", "Brindisi", "Foggia", "Lecce", "Taranto", 
    "Matera", "Potenza", "Catanzaro", "Cosenza", "Crotone", "Reggio Calabria", "Vibo Valentia"]

panel_data = add_province_dummy(panel_data, north, "North")
panel_data = add_province_dummy(panel_data, centre, "Centre")
panel_data = add_province_dummy(panel_data, south, "South")
panel_data = add_province_dummy(panel_data, island, "Islands")

large_city = ["Roma", "Milano", "Napoli", "Torino", "Palermo", "Genova"]
med_city = [
    "Bologna", "Firenze", "Bari", "Catania", "Verona", 
    "Venezia", "Messina", "Padova", "Trieste", "Brescia", 
    "Parma", "Prato", "Taranto", "Modena", "Reggio Calabria", 
    "Reggio Emilia", "Perugia", "Ravenna", "Livorno", "Rimini", 
    "Cagliari", "Foggia", "Ferrara", "Latina", "Salerno",
    "Monza", "Sassari", "Bergamo", "Trento", 
    "Pescara", "Forlì-Cesena", "Siracusa", "Vicenza", "Terni", 
    "Bolzano", "Piacenza", "Novara"
]

city = large_city + med_city

#panel_data = add_province_dummy(panel_data, large_city, "Large City")
#panel_data = add_province_dummy(panel_data, med_city, "Medium City")
panel_data = add_province_dummy(panel_data, city, "City")

panel_data = add_VAR_region(panel_data, rmt_wrk, "Regional remote work rate")
panel_data["Regional remote work rate"] = (panel_data["Regional remote work rate"]).round(4)

# Create year-over-year difference for Regional remote work rate
panel_data = panel_data.sort_values(['Province', 'Year'])
#panel_data['Regional remote work rate_diff'] = (panel_data.groupby('Province')['Regional remote work rate'].diff().fillna(0)).round(4)



In [125]:

for df, name in var_list:
    df = transform_data(df)
    df = transform_float(df)
    # Divide df by 10 
    df[df.columns[1:]] = df[df.columns[1:]].apply(lambda x: x/10)
    panel_data = add_VAR_province(panel_data, df, name)

panel_data["Population density"] = (panel_data["Population density"]/10 ).round(2)
panel_data["Short-term rental per capita"] = (panel_data["Short-term rental per capita"]/10 ).round(2)
panel_data["Unemployment rate"] = (panel_data["Unemployment rate"]/100 ).round(2)
panel_data["Population"] = (panel_data["Population"]*10 )
panel_data["Population"] = panel_data["Population"].astype(int)
# panel_data['Unemployment_diff'] = (panel_data.groupby('Province')['Unemployment rate'].diff().fillna(0)).round(4)

In [126]:
missing_data = panel_data.isnull().sum()
missing_data

Province                        0
Year                            0
Affordability Index             0
Ln affordability Index          0
Region                          0
Median net stipend              0
Average rent                    0
Mortgage rate                   0
North                           0
Centre                          0
South                           0
Islands                         0
City                            0
Regional remote work rate       0
Population density              0
Short-term rental per capita    0
Total Migration per 1k          0
Population                      0
Unemployment rate               0
dtype: int64

In [127]:
#show rows with missing data
missing_rows = panel_data[panel_data.isnull().any(axis=1)]
print(missing_rows)

Empty DataFrame
Columns: [Province, Year, Affordability Index, Ln affordability Index, Region, Median net stipend, Average rent, Mortgage rate, North, Centre, South, Islands, City, Regional remote work rate, Population density, Short-term rental per capita, Total Migration per 1k, Population, Unemployment rate]
Index: []


In [128]:
panel_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 642 entries, 0 to 641
Data columns (total 19 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Province                      642 non-null    str    
 1   Year                          642 non-null    int64  
 2   Affordability Index           642 non-null    float64
 3   Ln affordability Index        642 non-null    float64
 4   Region                        642 non-null    str    
 5   Median net stipend            642 non-null    float64
 6   Average rent                  642 non-null    int64  
 7   Mortgage rate                 642 non-null    float64
 8   North                         642 non-null    int64  
 9   Centre                        642 non-null    int64  
 10  South                         642 non-null    int64  
 11  Islands                       642 non-null    int64  
 12  City                          642 non-null    int64  
 13  Regional remote 

In [129]:
panel_data.head(6)

,Province,Year,Affordability Index,Ln affordability Index,Region,Median net stipend,Average rent,Mortgage rate,North,Centre,South,Islands,City,Regional remote work rate,Population density,Short-term rental per capita,Total Migration per 1k,Population,Unemployment rate
0,Agrigento,2018,0.27,-1.309333,Sicilia,1410.14,384,0.0666,0,0,0,1,0,0.0272,141.56,0.15,-6.6,432335,0.27
1,Agrigento,2019,0.29,-1.237874,Sicilia,1428.75,416,0.0659,0,0,0,1,0,0.0340,140.15,0.17,-5.0,428003,0.24
2,Agrigento,2020,0.28,-1.272966,Sicilia,1446.12,408,0.0629,0,0,0,1,0,0.1800,138.67,0.12,-3.3,423488,0.22
3,Agrigento,2021,0.34,-1.078810,Sicilia,1446.12,488,0.0636,0,0,0,1,0,0.1350,136.27,0.18,-1.6,416181,0.22
4,Agrigento,2022,0.31,-1.171183,Sicilia,1482.32,456,0.0695,0,0,0,1,0,0.0741,136.18,0.39,-1.4,415887,0.19
5,Agrigento,2023,0.34,-1.078810,Sicilia,1495.07,504,0.0852,0,0,0,1,0,0.0630,135.29,0.55,-1.7,413177,0.16


In [130]:
panel_data.to_csv("panel_data.csv", index=False, encoding='utf-8')